# Vector Database Landscape — Hands-On

Offline scoring matrix and ANN recall simulation.

## 0. Selection matrix

In [ ]:
%pip install -q numpy
stores={"FAISS":{"managed":0,"filters":0,"hybrid":0,"simple":2},"Chroma":{"managed":0,"filters":1,"hybrid":1,"simple":3},"pgvector":{"managed":1,"filters":3,"hybrid":2,"simple":2},"Qdrant":{"managed":2,"filters":3,"hybrid":2,"simple":2},"Weaviate":{"managed":2,"filters":2,"hybrid":3,"simple":2},"Pinecone":{"managed":3,"filters":2,"hybrid":2,"simple":2},"Milvus":{"managed":1,"filters":2,"hybrid":2,"simple":1}}

## 1. Score by requirements

In [ ]:
def rank(weights):
    return sorted([(sum(v.get(k,0)*w for k,w in weights.items()), name) for name,v in stores.items()], reverse=True)
for weights in [{"simple":3},{"managed":3,"filters":2},{"filters":3,"hybrid":2}]:
    print(weights, "->", rank(weights)[:3])

## 2. Exact vector search baseline

In [ ]:
import numpy as np
rng=np.random.RandomState(0)
vecs=rng.randn(500,24); vecs/=np.linalg.norm(vecs,axis=1,keepdims=True)
q=rng.randn(24); q/=np.linalg.norm(q)
exact=list(np.argsort(vecs@q)[-10:][::-1])
print(exact[:5])

## 3. Approximate search recall simulation

In [ ]:
def approx(frac, seed):
    r=np.random.RandomState(seed); n=max(10,int(len(vecs)*frac)); cand=r.choice(len(vecs), n, replace=False)
    found=list(cand[np.argsort(vecs[cand]@q)[-10:]])
    return len(set(found)&set(exact))/10
for frac in [0.05,0.1,0.25,0.5,1.0]:
    vals=[approx(frac,s) for s in range(5)]
    print(frac, round(float(np.mean(vals)),2))

## 4. Filter selectivity affects candidate pools

In [ ]:
tenants=np.array([i%5 for i in range(len(vecs))])
def filtered_exact(tenant):
    idx=np.where(tenants==tenant)[0]
    ranked=idx[np.argsort(vecs[idx]@q)[-5:][::-1]]
    return list(ranked)
print("tenant 2 top", filtered_exact(2))

## 5. Exercise prompts
1. Add cost and data-residency weights.
2. Simulate quantization by rounding vectors.
3. Compare filtered vs unfiltered recall.